# Zero noise vs O4b noise: medians and credible intervals

Compares a zero-noise run with a run in one O4b noise realization. For each configuration
found in both folders, the notebook prints the median and the 68% credible interval of every
sampled parameter in the two runs, and checks whether the injected value falls inside the
interval. The same numbers are also printed as rows of a LaTeX table.

Run from `bilby_inference/scripts/`.

In [ ]:
import os, sys, glob, re
import numpy as np

SCRIPT_DIR = os.getcwd()                     # .../bilby_inference/scripts
BILBY_DIR = os.path.dirname(SCRIPT_DIR)
PROJECT_DIR = os.path.dirname(BILBY_DIR)
sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, BILBY_DIR)

import bilby
bilby.core.utils.log.setup_logger(log_level="WARNING")

# Edit here (folders relative to bilby_inference/)
ZERO_DIR = os.path.join("zero_noise", "outdir_fig2")            # zero-noise reference run
NOISE_DIR = os.path.join("with_noise", "outdir_fig2_O4b_01")    # real-noise run (one realization)
SELECT = ""                                                     # optional substring of the labels, e.g. "5D_q+0.00"

NOISE_TAG = re.compile(r"_O4b\d{2}")

In [ ]:
def results_by_key(directory):
    """{label without noise tag: path} for every *_result.json in `directory` containing SELECT."""
    out = {}
    for path in sorted(glob.glob(os.path.join(BILBY_DIR, directory, "*_result.json"))):
        label = os.path.basename(path)[:-len("_result.json")]
        if SELECT in label:
            out[NOISE_TAG.sub("", label)] = path
    return out


def summary(result, name):
    """Median, upper and lower errors (84th-50th, 50th-16th percentiles) and 68% width of a marginal."""
    low, med, high = np.percentile(np.asarray(result.posterior[name]), [16, 50, 84])
    return med, high - med, med - low, high - low


def fmt(med, plus, minus, digits=3):
    """Median with asymmetric errors, as text and as LaTeX."""
    text = f"{med:+.{digits}f} +{plus:.{digits}f} -{minus:.{digits}f}"
    latex = f"${med:.{digits}f}^{{+{plus:.{digits}f}}}_{{-{minus:.{digits}f}}}$"
    return text, latex


zero_files, noise_files = results_by_key(ZERO_DIR), results_by_key(NOISE_DIR)
common = [key for key in zero_files if key in noise_files]
print(f"{len(common)} pairs; only zero noise: {sorted(set(zero_files) - set(noise_files))}; "
      f"only O4b: {sorted(set(noise_files) - set(zero_files))}")

for key in common:
    zero = bilby.result.read_in_result(zero_files[key])
    noise = bilby.result.read_in_result(noise_files[key])
    print(f"\n{key}")
    print(f"  zero : {os.path.basename(zero_files[key])}")
    print(f"  noise: {os.path.basename(noise_files[key])}")
    print(f"  {'param':<7} {'inj':>8} | {'zero (68% CI)':>26} {'w':>6} | {'O4b (68% CI)':>26} {'w':>6} | {'in?':>7}")
    latex_rows = []
    for name in zero.search_parameter_keys:
        inj = zero.injection_parameters[name]
        z_med, z_plus, z_minus, z_w = summary(zero, name)
        n_med, n_plus, n_minus, n_w = summary(noise, name)
        z_txt, z_tex = fmt(z_med, z_plus, z_minus)
        n_txt, n_tex = fmt(n_med, n_plus, n_minus)
        # is the injected value inside the 68% interval? (zero / O4b)
        z_in = "y" if z_med - z_minus <= inj <= z_med + z_plus else "n"
        n_in = "y" if n_med - n_minus <= inj <= n_med + n_plus else "n"
        print(f"  {name:<7} {inj:8.3f} | {z_txt:>26} {z_w:6.3f} | {n_txt:>26} {n_w:6.3f} | {z_in:>3}/{n_in}")
        latex_rows.append(f"{name} & {inj:.3f} & {z_tex} & {n_tex} \\\\")
    print("  LaTeX:")
    print("\n".join("    " + row for row in latex_rows))